# 07 — Publication & Net-Volume Reconciliation

Reconciles every demand and supply leg into the **single official, versioned net volume** described in
[`../specifications/07-publication-net-volume.md`](../specifications/07-publication-net-volume.md).

**Depends on 01, 03, 05, 06** (and 04 for the BTM reconciliation). Run after them.

**Capability tables**
- `volume_forecast_gold_net_volume` (source of truth — versioned per interval)
- `volume_forecast_gold_publication_log` (publication lifecycle / SLA)
- `volume_forecast_gold_consumer_handoff` (contract for downstream consumers)

**Reconciliation:** `net_volume = (consumption + industrial) − (utility wind + utility solar)`. Behind-the-meter PV
is **already netted into demand by 04**, so it is surfaced for audit but never re-added to supply (no double count).
Two versions are emitted per interval (`v1` SUPERSEDED → `v2` PUBLISHED) so the revision story is visible.

**UC comments:** [`uc_table_comments.py`](./uc_table_comments.py) — applied in the final cell.

In [ ]:
import os
import datetime as dt

from pyspark.sql import functions as F

dbutils.widgets.text("catalog", os.environ.get("DEMO_UC_CATALOG", "energy_utilities"))
dbutils.widgets.text("schema", os.environ.get("DEMO_UC_SCHEMA", "energy_trading2"))

CATALOG = dbutils.widgets.get("catalog").strip() or "energy_utilities"
SCHEMA = dbutils.widgets.get("schema").strip() or "energy_trading2"
print(f"Target: {CATALOG}.{SCHEMA}")
spark.sql(f"USE `{CATALOG}`.`{SCHEMA}`")


def fq(name: str) -> str:
    return f"`{CATALOG}`.`{SCHEMA}`.`{name}`"


TODAY = dt.date.today()
LATEST_DATE = TODAY
ZONES = ["DE", "NL", "FR", "BE", "AT"]
AS_OF_V1 = dt.datetime.combine(LATEST_DATE, dt.time(6, 0))
AS_OF_V2 = dt.datetime.combine(LATEST_DATE, dt.time(12, 0))

REQUIRED = [
    "volume_forecast_silver_consumption_st",
    "volume_forecast_silver_industrial_load",
    "volume_forecast_silver_wind_forecast",
    "volume_forecast_silver_solar_forecast",
    "volume_forecast_silver_btm_pv",
]
missing = [t for t in REQUIRED if not spark.catalog.tableExists(fq(t))]
assert not missing, f"Run upstream notebooks first — missing: {missing}"

In [ ]:
# ---- Reconcile all legs onto the (date, zone, interval) grid ----
keys = ["delivery_date", "zone_code", "interval_start"]

# Consumption carries two forecast vintages — reconcile only the latest run.
_cons_src = spark.table(fq("volume_forecast_silver_consumption_st"))
_cons_latest_ts = _cons_src.agg(F.max("forecast_ts").alias("m")).first()["m"]
cons = (_cons_src.filter(F.col("forecast_ts") == F.lit(_cons_latest_ts))
        .groupBy(*keys).agg(F.sum("p50_mw").alias("consumption_st_mw")))
ind = spark.table(fq("volume_forecast_silver_industrial_load")).groupBy(*keys).agg(F.sum("baseline_mw").alias("industrial_mw"))
wind = spark.table(fq("volume_forecast_silver_wind_forecast")).groupBy(*keys).agg(F.sum("p50_mw").alias("wind_mw"))
solar = spark.table(fq("volume_forecast_silver_solar_forecast")).groupBy(*keys).agg(F.sum("p50_mw").alias("solar_mw"))
btm = spark.table(fq("volume_forecast_silver_btm_pv")).groupBy(*keys).agg(F.sum("btm_pv_mw").alias("btm_pv_mw"))

base = cons.join(ind, keys, "outer").join(wind, keys, "outer").join(solar, keys, "outer").join(btm, keys, "outer")
for c in ["consumption_st_mw", "industrial_mw", "wind_mw", "solar_mw", "btm_pv_mw"]:
    base = base.withColumn(c, F.round(F.coalesce(F.col(c), F.lit(0.0)), 2))

base = (base
    .withColumn("total_demand_mw", F.round(F.col("consumption_st_mw") + F.col("industrial_mw"), 2))
    .withColumn("total_supply_mw", F.round(F.col("wind_mw") + F.col("solar_mw"), 2))
    .withColumn("net_volume_mw", F.round(F.col("consumption_st_mw") + F.col("industrial_mw") - F.col("wind_mw") - F.col("solar_mw"), 2)))

LEG_COLS = ["consumption_st_mw", "industrial_mw", "total_demand_mw", "wind_mw", "solar_mw", "btm_pv_mw", "total_supply_mw"]
SELECT_COLS = (["delivery_date", "interval_start", "zone_code", "publication_version"] + LEG_COLS
               + ["net_volume_mw", "prev_published_mw", "revision_mw", "publication_status", "as_of_ts"])

# v1: an earlier cut (2% lighter net), later SUPERSEDED.
v1 = (base
    .withColumn("net_volume_mw", F.round(F.col("net_volume_mw") * 0.98, 2))
    .withColumn("publication_version", F.lit(1))
    .withColumn("prev_published_mw", F.lit(None).cast("double"))
    .withColumn("revision_mw", F.lit(None).cast("double"))
    .withColumn("publication_status", F.lit("SUPERSEDED"))
    .withColumn("as_of_ts", F.lit(AS_OF_V1).cast("timestamp"))
    .select(*SELECT_COLS))

# v2: the PUBLISHED official version, carrying the revision vs v1.
v2 = (base
    .withColumn("publication_version", F.lit(2))
    .withColumn("prev_published_mw", F.round(F.col("net_volume_mw") * 0.98, 2))
    .withColumn("revision_mw", F.round(F.col("net_volume_mw") - F.col("net_volume_mw") * 0.98, 2))
    .withColumn("publication_status", F.lit("PUBLISHED"))
    .withColumn("as_of_ts", F.lit(AS_OF_V2).cast("timestamp"))
    .select(*SELECT_COLS))

net_volume = v1.unionByName(v2)
net_volume.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(fq("volume_forecast_gold_net_volume"))

print("net_volume:", spark.table(fq("volume_forecast_gold_net_volume")).count())
display(spark.table(fq("volume_forecast_gold_net_volume"))
        .filter((F.col("publication_status") == "PUBLISHED") & (F.col("delivery_date") == F.lit(LATEST_DATE)))
        .orderBy("zone_code", "interval_start").limit(20))

In [ ]:
# ---- Publication log: lifecycle / SLA per (date, zone, version) ----
from pyspark.sql import Row

nv = spark.table(fq("volume_forecast_gold_net_volume"))
leg_flag_cols = ["consumption_st_mw", "industrial_mw", "wind_mw", "solar_mw", "btm_pv_mw"]

log = (nv.groupBy("delivery_date", "zone_code", "publication_version").agg(
        F.countDistinct("interval_start").alias("n_int"),
        F.first("as_of_ts").alias("published_ts"),
        F.first("publication_status").alias("status"),
        *[F.max(F.when(F.col(c) > 0, 1).otherwise(0)).alias(c + "_p") for c in leg_flag_cols],
    )
    .withColumn("legs_present", sum(F.col(c + "_p") for c in leg_flag_cols))
    .withColumn("completeness_pct", F.round(F.col("n_int") / 96.0 * 100.0, 1))
    .withColumn("gate_status", F.when(F.col("completeness_pct") < 100.0, F.lit("BLOCKED"))
                                .when(F.col("publication_version") == 2, F.lit("PUBLISHED")).otherwise(F.lit("PUBLISHED")))
    .withColumn("superseded_version", F.when(F.col("publication_version") == 2, F.lit(1)).otherwise(F.lit(None).cast("int")))
    .withColumn("note", F.when(F.col("legs_present") < 5, F.concat(F.lit("Only "), F.col("legs_present").cast("string"), F.lit(" of 5 legs present (zone has no asset for some legs)")))
                         .otherwise(F.lit("All 5 legs reconciled")))
    .select("delivery_date", "zone_code", "publication_version", "published_ts", "legs_present",
            "completeness_pct", "gate_status", "superseded_version", "note"))

log.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(fq("volume_forecast_gold_publication_log"))

# ---- Consumer handoff: contract per downstream consumer x zone ----
CONSUMERS = [
    ("SHORT_TERM_SQUARING", "net_volume_mw", "INTRADAY", 15),
    ("LONG_TERM_CURVE",     "net_volume_mw", "SEASONAL", 1440),
    ("DSR_BIDSTACK",        "net_volume_mw", "INTRADAY", 30),
    ("RISK",                "net_volume_mw", "DAILY", 240),
    ("REPORTING",           "net_volume_mw", "DAILY", 720),
]
handoff_rows = [
    Row(consumer=c, zone_code=z, handoff_field=field, cadence=cadence, latest_version=2, sla_minutes=sla)
    for (c, field, cadence, sla) in CONSUMERS for z in ZONES
]
spark.createDataFrame(handoff_rows).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(fq("volume_forecast_gold_consumer_handoff"))

display(spark.table(fq("volume_forecast_gold_publication_log")).orderBy(F.col("delivery_date").desc(), "zone_code", "publication_version"))

In [ ]:
# Row counts + Unity Catalog comments.
for t in [
    "volume_forecast_gold_net_volume",
    "volume_forecast_gold_publication_log",
    "volume_forecast_gold_consumer_handoff",
]:
    print(f"  {t:44s}  {spark.table(fq(t)).count():>10,} rows")

from pathlib import Path

_uc_paths = []
try:
    _nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    _uc_paths.append(Path(_nb).parent / "uc_table_comments.py")
except Exception:
    pass
_uc_paths.append(Path.cwd() / "uc_table_comments.py")

_uc_py = next((p for p in _uc_paths if p.is_file()), None)
if _uc_py is None:
    raise FileNotFoundError("uc_table_comments.py not found next to this notebook.")

exec(_uc_py.read_text(), globals())
apply_volume_forecast_notebook_07_comments(spark, CATALOG, SCHEMA)
print("UC comments applied for notebook 07.")